In [1]:
import os
os.getcwd()

'/home/ruchir321/Documents/repositories/aigent/claude-cookbooks/agents'

In [2]:
os.chdir("../../")
os.getcwd()

'/home/ruchir321/Documents/repositories/aigent'

In [3]:
from concurrent.futures import ThreadPoolExecutor
from utils.utils import llm_call, extract_xml
from typing import List, Dict, Callable
from langfuse import observe, get_client

In [4]:
import dotenv
dotenv.load_dotenv()

os.environ["LANGFUSE_SECRET_KEY"] = dotenv.dotenv_values()["LANGFUSE_SECRET_KEY"]
os.environ["LANGFUSE_PUBLIC_KEY"] = dotenv.dotenv_values()["LANGFUSE_PUBLIC_KEY"]
os.environ["LANGFUSE_HOST"] = dotenv.dotenv_values()["LANGFUSE_HOST"]

In [5]:
@observe
def chain(input: str, prompts: List[str]):
    """Chain multiple LLM calls sequentially passing results between steps

    input --> [LLM1] --> result1 --> [LLM2] --> result2 --> ... --> response to user"""
    result = input
    for i, prompt in enumerate(prompts, start=1):
        print(f"\n Step {i}:")
        result = llm_call(prompt=f"{prompt}\nInput: {result}")
        print(result.message.content)
    
    return result

@observe
def parallel(prompt: str, inputs: List[str], n_workers: int = 3) -> List[str]:
    """Process multiple inputs concurrently with the same prompt (???)"""
    with ThreadPoolExecutor(max_workers=n_workers) as executor:
        futures = [executor.submit(llm_call,f"{prompt}\nInput: {x}") for x in inputs]
        return [f.result() for f in futures]
    
@observe
def route(input: str, routes: Dict[str, str]) -> str:
    """Route inputs to specialized prompts using content classification"""
    print(f"\nAvailable routes: {list(routes.keys)}")
    selector_prompt = f"""
    Analyze the input and select the most appropriate support team from these options: {list(routes.keys())}
    First explain your reasoning, then provide your selection in this XML format:

    <reasoning>
    Brief explanation of why this ticket should be routed to a specific team.
    Consider key terms, user intent, and urgency level.
    </reasoning>

    <selection>
    The chosen team name
    </selection>

    Input: {input}"""

    route_response = llm_call(prompt=selector_prompt)
    reasoning = extract_xml(text = route_response, tag="reasoning")
    route_key = extract_xml(text= route_response, tag="selection").strip().lower()

    print("Routing Analysis: ")
    print(reasoning)
    print(f"\nSelected routes: {route_key}")

    # Process input with selected specialized prompts
    selected_prompt = routes[route_key]
    return llm_call(prompt=f"{selected_prompt}\nInput: {input}")



# Examples

## 1: Chain workflows for structured data extraction and formatting

Each step will progressively transform raw input into a formatted table

In [6]:
data_processing_steps = [
    """Extract only the numerical values and their associated metrics from the text.
    Format each as 'value: metric' on a new line.
    Example format:
    92: customer satisfaction
    45%: revenue growth""",

    """Add negative(-) or positive(+) sign for decrease or increase in values respectively
    Example format:
    -5%: revenue growth
    """,
    
    """Convert all numerical values to percentages where possible.
    If not a percentage or points, convert to decimal (e.g., 92 points -> 92%).
    Keep one number per line.
    Example format:
    92%: customer satisfaction
    45%: revenue growth""",
    
    """Sort all lines in descending order by numerical value.
    Keep the format 'value: metric' on each line.
    Example:
    92%: customer satisfaction
    87%: employee satisfaction""",
    
    """Format the sorted data as a markdown table with columns:
    | Metric | Value |
    |:--|--:|
    | Customer Satisfaction | 92% |"""
]

report = """
Q3 Performance Summary:
Our customer satisfaction score rose to 42 points this quarter.
Revenue shrunk by 5% compared to last year.
Market share is now at 23% in our primary market.
Customer churn decreased to 5% from 8%.
New user acquisition cost is $43 per user.
Product adoption rate increased to 78%.
Employee satisfaction is at 87 points.
Operating margin improved to 34%.
"""

langfuse = get_client()
print("\nInput text:")
print(report)
formatted_result = chain(report, data_processing_steps)
langfuse.flush()


Input text:

Q3 Performance Summary:
Our customer satisfaction score rose to 42 points this quarter.
Revenue shrunk by 5% compared to last year.
Market share is now at 23% in our primary market.
Customer churn decreased to 5% from 8%.
New user acquisition cost is $43 per user.
Product adoption rate increased to 78%.
Employee satisfaction is at 87 points.
Operating margin improved to 34%.


 Step 1:
42: customer satisfaction  
5: revenue growth  
23: market share  
5: customer churn  
43: new user acquisition cost  
78: product adoption rate  
87: employee satisfaction  
34: operating margin

 Step 2:
-42: customer satisfaction  
-5: revenue growth  
+23: market share  
-5: customer churn  
+43: new user acquisition cost  
+78: product adoption rate  
+87: employee satisfaction  
+34: operating margin

 Step 3:
42%: customer satisfaction  
-5%: revenue growth  
+23%: market share  
-5%: customer churn  
+43%: new user acquisition cost  
+78%: product adoption rate  
+87%: employee sati

## 2: Parallelization workflow for group analysis

Process impact analysis for multiple stakeholder groups concurrently

In [ ]:
stakeholders = [
    """Customers:
    - Price sensitive
    - Want better tech
    - Environmental concerns""",
    
    """Employees:
    - Job security worries
    - Need new skills
    - Want clear direction""",
    
    """Investors:
    - Expect growth
    - Want cost control
    - Risk concerns""",
    
    """Suppliers:
    - Capacity constraints
    - Price pressures
    - Tech transitions"""
]

impact_results = parallel(
    """Analyze how market changes will impact this stakeholder group.
    Provide specific impacts and recommended actions.
    Format with clear sections and priorities.""",
    stakeholders
)

for result in impact_results:
    print(result.message.content)
    print('+' * 80)